In [2]:
import pandas as pd
import numpy as np

In [3]:
df = pd.read_csv('/Users/alexren/Desktop/UC Davis/Stats/STA 141B/Project/cleaned_crime_data(1).csv')

In [4]:
df.head()

,caseNumber,reportDate,offenseDescription1,offenseDescription2,offenseDescription3,offenseDescription4,offenseDescription5,offenseDescription6,incidentNumber,place,...,disposition,beat,xCoord,yCoord,blockNumber,street,coordCheck,uploadDate,crimeType,crimeClassification
0,21-01994,2021-05-17T00:00:00,Contact minor with intent sex (F),Annoy/molest victim under 18 years of age - Di...,NaN,NaN,NaN,NaN,202105170050,HOLMES JUNIOR HIGH SCHOOL,...,C,2,-121.739891,38.557056,1200,DREXEL DR,Y,2025-03-25T09:00:17.477,All Other Offenses,Felony
1,21-04672,2021-11-04T00:00:00,Grand theft:money/labor/property - Shoplifting...,Conspiracy:commit crime (F),Possess narcotic controlled substance (M),Bench warrant/failure to appear on felony char...,Bench warrant/failure to appear on misdemeanor...,Shoplifting (M),202111040139,NUGGET MARKET,...,A,2,-121.733101,38.560970,1400,E COVELL BL,Y,2024-10-02T21:00:02.427,Larceny/Theft,Felony
2,21-04680,2021-11-05T00:00:00,Rape by force/fear/etc - Rape (F),Mandated Reporter/Child Abuse,NaN,NaN,NaN,NaN,202111050086,DAVIS HIGH SCHOOL,...,E,2,-121.750435,38.555664,300,W 14TH ST,Y,2025-09-05T10:25:02.48,Sex Offenses-Forcible,Felony
3,21-04689,2021-11-06T00:00:00,Burglary:second degree - Burglary/breaking and...,NaN,NaN,NaN,NaN,NaN,202111060040,PKG HOLDINGS,...,NaN,4,-121.687393,38.553787,5100,CHILES RD,Y,2025-04-22T16:35:40.863,Burglary,Felony
4,21-04769,2021-11-10T00:00:00,Battery on person - Simple (M),Theft of personal property - From motor vehicl...,NaN,NaN,NaN,NaN,202111100119,CORK IT AGAIN,...,A,2,-121.739601,38.546055,800,4TH ST,Y,2025-04-22T16:35:40.863,Assault,Misdemeanor


In [5]:
offense_cols = ['offenseDescription1', 'offenseDescription2', 'offenseDescription3', 'offenseDescription4',
                    'offenseDescription5', 'offenseDescription6']

df['offense_description'] = df[offense_cols].apply(lambda x: '; '.join(x.dropna()), axis=1)


In [6]:
df = df.drop(columns = offense_cols)

In [7]:
cols = list(df.columns)

In [8]:
cols.remove('offense_description')

In [9]:
cols.insert(cols.index('reportDate') + 1, 'offense_description')

In [10]:
cols.remove('occurrence1Date')

In [11]:
cols.insert(cols.index('reportDate') + 1, 'occurrence1Date')

In [12]:
cols.remove('occurrence1Time')

In [13]:
cols.insert(cols.index('occurrence1Date') + 1, 'occurrence1Time')

In [14]:
df = df[cols]

In [15]:
def clean_col_names(name):
    new_name = ''
    
    for char in name:
        if char.isupper():
            new_name += ("_" + char.lower())
        else:
            new_name += char
            
    return new_name

In [16]:
df.columns = [clean_col_names(col) for col in df.columns]

In [17]:
from datetime import datetime

In [18]:
df['report_date'] = df['report_date'].str.replace('T00:00:00', '')
df['report_date'] = pd.to_datetime(df['report_date'])

In [19]:
df = df.rename(columns = {'occurrence1_date': 'occurrence_date', 'occurrence1_time': 'occurrence_time'})

In [20]:
df['occurrence_date'] = df['occurrence_date'].str.replace('T00:00:00', '')
df['occurrence_date'] = pd.to_datetime(df['occurrence_date'])

In [21]:
df['upload_date'] = df['upload_date'].str.split('T').str[0]
df['upload_date'] = pd.to_datetime(df['upload_date'])

In [22]:
correct_times = []

for time in df['occurrence_time']:
    time = str(time).zfill(4)
    
    hour = int(time[0:2])
    minutes_tens = int(time[2])
    minutes_ones = int(time[3])

    if minutes_tens >= 6:
        hour += 1
        minutes_tens = minutes_tens - 6

    if hour == 24:
        hour = 0

    correct_times.append(f'{hour:02d}:{minutes_tens}{minutes_ones}')

In [23]:
df['occurrence_time'] = correct_times

In [24]:
df = df.sort_values(by=['occurrence_date', 'occurrence_time'])

In [25]:
df['place'] = df['place'].fillna('')

In [26]:
df = df[df['city'] == 'Davis']

In [27]:
df.head(10)

,case_number,report_date,occurrence_date,occurrence_time,offense_description,incident_number,place,city,disposition,beat,x_coord,y_coord,block_number,street,coord_check,upload_date,crime_type,crime_classification
2790,24-04377,2024-10-21,2020-10-21,22:00,Battery on person - Simple (M),202410210178,,Davis,NaN,2,-121.744010,38.545210,0,4TH ST/C ST,Y,2025-04-23,Assault,Misdemeanor
428,22-05316,2022-11-28,2021-01-01,00:00,Grand theft:money/labor/property - Motor vehic...,202211280021,,Davis,NaN,1,-121.755478,38.562660,2200,FORTUNA CT,Y,2026-01-30,Larceny/Theft,Felony
20,21-05281,2021-12-12,2021-03-16,00:00,Stalking (F),202112120047,PACIFICO COOPERATIVE COMMUNITY,Davis,NaN,4,-121.726723,38.537979,1700,DREW CI,Y,2025-01-23,Assault,Felony
0,21-01994,2021-05-17,2021-03-18,12:07,Contact minor with intent sex (F); Annoy/moles...,202105170050,HOLMES JUNIOR HIGH SCHOOL,Davis,C,2,-121.739891,38.557056,1200,DREXEL DR,Y,2025-03-25,All Other Offenses,Felony
2,21-04680,2021-11-05,2021-05-01,00:00,Rape by force/fear/etc - Rape (F); Mandated Re...,202111050086,DAVIS HIGH SCHOOL,Davis,E,2,-121.750435,38.555664,300,W 14TH ST,Y,2025-09-05,Sex Offenses-Forcible,Felony
19,21-05238,2021-12-09,2021-05-05,00:00,Rape:victim incapable of giving consent (F); S...,202112090022,WAKE FOREST APARTMENTS,Davis,NaN,1,-121.763283,38.548244,1300,WAKE FOREST DR,Y,2025-04-23,Sex Offenses-Forcible,Felony
46,22-00167,2022-01-12,2021-05-06,20:30,Get credit/etc others id - False pretenses/swi...,202201120039,,Davis,NaN,1,-121.769028,38.549236,700,ADAMS ST,Y,2025-11-12,Fraud Offenses,Felony
60,22-00479,2022-02-02,2021-07-01,00:00,Get credit/etc others id - Identity theft (F);...,202202020072,BANK OF AMERICA,Davis,NaN,3,-121.741257,38.544838,300,E ST,Y,2025-11-12,Fraud Offenses,Felony
2784,24-04368,2024-10-21,2021-07-01,10:00,Get credit/etc others id - False pretenses/swi...,202410210076,ADAMS STREET COMPLEX,Davis,NaN,1,-121.769028,38.549236,700,ADAMS ST,Y,2025-12-01,Fraud Offenses,Felony
83,22-00835,2022-02-24,2021-07-23,13:25,Obtain money/etc by false pretenses [over $400...,202202240061,,Davis,NaN,1,-121.786003,38.559719,1900,LAKE BL,Y,2025-11-12,Fraud Offenses,Felony


In [28]:
df.to_csv('cleaned_crime_data_revised.csv', index = False)